# Estimating a Residential Energy Demand System with Seemingly Unrelated Regression

## Executive Summary

A regional utility estimates a two-equation residential **energy demand system** — electricity and natural gas budget shares as functions of own-price, cross-price, and household income — using **PROC SYSLIN** with the seemingly-unrelated-regression (SUR) method. The two share equations share a common household budget, so their errors are cross-correlated; SUR estimates the equations jointly to exploit that correlation, recovering sign-coherent own- and cross-price effects and supplying the cross-equation covariance and restriction tests a demand analyst needs.

## Data Sources

| Dataset | Rows | Grain | Key Variables | Description |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | one row per monthly market observation | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Synthetic monthly residential energy market panel. `lp_elec`/`lp_gas` are log real prices of electricity and natural gas; `lincome` is log real household income; `w_elec`/`w_gas` are expenditure budget shares for electricity and natural gas, generated from a known AIDS-style demand structure plus correlated cross-equation noise. |

# Estimating a Residential Energy Demand System with Seemingly Unrelated Regression

A regional gas-and-electric utility wants to understand how households reallocate their energy budget between **electricity** and **natural gas** as relative prices and income change. The natural framework is a **demand system**: a set of budget-share equations estimated jointly.

Two features make joint estimation the right tool:

- The share equations draw on a common household budget, so their errors are **cross-correlated** — when a household spends more on electricity it spends less on gas. Estimating the equations together with **seemingly unrelated regression (SUR)** uses that correlation for efficiency.
- Economic theory imposes **linear restrictions** (adding-up, homogeneity, symmetry) that a system estimator can enforce directly.

`PROC SYSLIN` with the `SUR` option is built for exactly this situation. It fits each share equation, estimates the cross-equation error covariance, and re-fits the system with that covariance to deliver efficient, theory-coherent elasticities — together with the cross-model covariance matrix and joint restriction tests.

In this notebook we:

1. Generate a synthetic monthly energy market panel from a known share structure.
2. Fit a two-equation share system with SUR.
3. Compare the joint SUR fit with equation-by-equation OLS.
4. Impose a homogeneity restriction and read its joint F-test.
5. Extract the coefficient table for elasticity reporting.

## Step 1 — Generate a synthetic energy market panel

We simulate 60 monthly observations of a small **two-good energy demand system** (electricity and natural gas). The data-generating process is a linearized, AIDS-style budget-share model:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

We deliberately build in **cross-equation correlation**: when households spend more on electricity they spend less on gas, so `e1` and `e2` are negatively correlated. A common energy-market price factor also drives both log-prices together. These are the features that reward joint SUR estimation over fitting each equation alone.

In [1]:
data energy;
   call streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   do month = 1 to 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      output;
   end;

   keep month lp_elec lp_gas lincome w_elec w_gas;
run;

proc means data=energy n mean std min max maxdec=3;
   var w_elec w_gas lp_elec lp_gas lincome;
run;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 2 — Baseline SUR estimation of the demand system

We estimate both share equations jointly. The `SUR` option tells `PROC SYSLIN` to estimate the cross-equation error covariance and use it for an efficient feasible-GLS fit. Two labeled `MODEL` statements (`elec` and `gas`) define the system; each regresses a budget share on the two log-prices and log-income.

SYSLIN reports each equation's parameter estimates and, at the end, the **Cross-Model Covariance Matrix** — the estimated covariance of the two equations' errors that SUR exploits.

In [2]:
proc syslin data=energy sur;
   elec: model w_elec = lp_elec lp_gas lincome;
   gas:  model w_gas  = lp_elec lp_gas lincome;
run;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Step 3 — Compare with equation-by-equation OLS

To see what SUR buys us, we re-fit the same two equations with ordinary least squares (the default method, one equation at a time). OLS ignores the cross-equation error correlation, so it is consistent but less efficient than SUR when that correlation is non-zero — as it is here by construction.

Comparing the two coefficient tables shows how the estimates and their standard errors shift once the system structure is taken into account.

In [3]:
proc syslin data=energy ols;
   elec: model w_elec = lp_elec lp_gas lincome;
   gas:  model w_gas  = lp_elec lp_gas lincome;
run;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Step 4 — Impose economic theory and test it

Demand theory says nominal price/income effects should obey **homogeneity of degree zero**: scaling all prices and income leaves real demand unchanged, so the price and income coefficients within an equation sum to zero. We impose that on the electricity share with a `RESTRICT` statement.

SYSLIN re-fits the system subject to the constraint and reports a **Restriction Results** F-test of whether the restriction is consistent with the data — a direct, joint test of the homogeneity hypothesis.

In [4]:
proc syslin data=energy sur;
   elec: model w_elec = lp_elec lp_gas lincome;
   gas:  model w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
run;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Step 5 — Capture estimates for elasticity reporting

For a final report we persist the converged coefficients with `OUTEST=`. The resulting dataset carries one row per equation with the intercept and slope estimates plus fit statistics, which feed the utility's own- and cross-price elasticity calculations at sample-mean shares. We print it for the record.

In [5]:
proc syslin data=energy sur outest=demand_est;
   elec: model w_elec = lp_elec lp_gas lincome;
   gas:  model w_gas  = lp_elec lp_gas lincome;
run;

proc print data=demand_est noobs;
   title "SUR demand-system coefficient estimates";
run;
title;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interpreting the results

**Sign coherence.** The SUR estimates recover the substitution structure built into the data. In the electricity-share equation the **own log-price coefficient is positive** (`LP_ELEC` = 0.112) while the **cross-price coefficient is negative** (`LP_GAS` = -0.098); in the gas-share equation the pattern mirrors it (own `LP_GAS` = 0.114, cross `LP_ELEC` = -0.075). Every own- and cross-price effect is statistically significant (each `Pr > |t|` below 0.005), so the substitution signs are firmly identified rather than artifacts of a noisy fit.

**SUR exploits the cross-equation correlation.** The **Cross-Model Covariance Matrix** shows a negative off-diagonal (-0.000068): households trade electricity spending against gas spending, exactly as constructed. Because that covariance is non-zero, joint SUR estimation is more efficient than fitting each equation alone — the SUR standard errors in Step 2 are uniformly slightly smaller than the equation-by-equation OLS standard errors in Step 3 (for example, the gas `LP_ELEC` standard error falls from 0.0264 under OLS to 0.0255 under SUR), while the point estimates are unchanged. That tighter inference is the payoff of modeling the system rather than two separate regressions.

**Theory holds up.** Imposing **homogeneity of degree zero** on the electricity share (price and income coefficients summing to zero) via `RESTRICT` produced a Restriction Results F-test of 2.14 with `Pr > F` = 0.149. The restriction is **not rejected**, so the homogeneity prediction of consumer-demand theory is consistent with this sample — the utility can carry the constrained, theory-coherent estimates into reporting with confidence.

**Operational use.** The `OUTEST=` table persists one row per equation with the intercept and slope estimates and fit statistics. Those coefficients convert directly into own- and cross-price elasticities and an income (expenditure) elasticity at the sample-mean shares (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 from Step 1), giving the utility a defensible, theory-consistent basis for demand forecasting and rate-design scenarios.